# Random Forest

In [12]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import confusion_matrix,classification_report,roc_auc_score


# Load Dataset

In [13]:
df=pd.read_csv("Customer-Churn.csv")


# Basic Cleaning

In [14]:
df=df.drop("customerID",axis=1)


In [15]:
df["TotalCharges"]=pd.to_numeric(df["TotalCharges"],errors="coerce")
df["TotalCharges"]=df["TotalCharges"].fillna(df["TotalCharges"].median())

# Encode

In [16]:
df["Churn"]=df["Churn"].map({"Yes":1,"No":0})

In [17]:
# Split

In [18]:
X=df.drop("Churn",axis=1)
y=df["Churn"]


In [52]:
cat_cols=X.select_dtypes(include="object").columns
num_cols=X.select_dtypes(exclude="object").columns

In [20]:
# Preprocessing

In [25]:
numeric_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="median"))
])
categorical_pipeline=Pipeline([                
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])
preprocessor=ColumnTransformer([
    ("num",numeric_pipeline,num_cols),
    ("cat",categorical_pipeline,cat_cols)
])


# Train Test Split

In [28]:
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42)

# Apply Preprocessing

In [30]:
X_train_prep=preprocessor.fit_transform(X_train)
X_test_prep=preprocessor.transform(X_test)


In [31]:
#Feature Selection

In [32]:
selector_model=RandomForestClassifier(
    n_estimators=300,
    random_state=42
)
selector_model.fit(X_train_prep,y_train)
selector=SelectFromModel(
    selector_model,
    threshold="median",
    prefit="True"
)
X_train_sel=selector.transform(X_train_prep)
X_test_sel=selector.transform(X_test_prep)
print("Original features:",X_train_prep.shape[1])
print("Selected features:",X_train_sel.shape[1])


Original features: 45
Selected features: 23


In [40]:
# FinalModel

In [34]:
model=RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    class_weight="balanced",
    random_state=42
)
model.fit(X_train_sel,y_train)



,n_estimators,500
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


# Predict


In [36]:
pred=model.predict(X_test_sel)
prob=model.predict_proba(X_test_sel)[:,1]


# Evaluate

In [39]:
print("\n Confusion Matrix")
print(confusion_matrix(y_test,pred))
print("\n Classification Report")
print(classification_report(y_test,pred))
print("\n ROC AUC")
print("\n ROC AUC:",roc_auc_score(y_test,prob))


 Confusion Matrix
[[830 205]
 [117 257]]

 Classification Report
              precision    recall  f1-score   support

           0       0.88      0.80      0.84      1035
           1       0.56      0.69      0.61       374

    accuracy                           0.77      1409
   macro avg       0.72      0.74      0.73      1409
weighted avg       0.79      0.77      0.78      1409


 ROC AUC

 ROC AUC: 0.8391588519465758


In [47]:
ohe=preprocessor.named_transformers_["cat"]["encoder"]
cat_feature_names=ohe.get_feature_names_out(cat_cols)
ohe,cat_feature_names

(OneHotEncoder(handle_unknown='ignore'),
 array(['gender_Female', 'gender_Male', 'Partner_No', 'Partner_Yes',
        'Dependents_No', 'Dependents_Yes', 'PhoneService_No',
        'PhoneService_Yes', 'MultipleLines_No',
        'MultipleLines_No phone service', 'MultipleLines_Yes',
        'InternetService_DSL', 'InternetService_Fiber optic',
        'InternetService_No', 'OnlineSecurity_No',
        'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
        'OnlineBackup_No', 'OnlineBackup_No internet service',
        'OnlineBackup_Yes', 'DeviceProtection_No',
        'DeviceProtection_No internet service', 'DeviceProtection_Yes',
        'TechSupport_No', 'TechSupport_No internet service',
        'TechSupport_Yes', 'StreamingTV_No',
        'StreamingTV_No internet service', 'StreamingTV_Yes',
        'StreamingMovies_No', 'StreamingMovies_No internet service',
        'StreamingMovies_Yes', 'Contract_Month-to-month',
        'Contract_One year', 'Contract_Two year', 'Pape

In [48]:
all_features_names=list(num_cols) + list(cat_feature_names)
all_features_names

['SeniorCitizen',
 'tenure',
 'MonthlyCharges',
 'TotalCharges',
 'gender_Female',
 'gender_Male',
 'Partner_No',
 'Partner_Yes',
 'Dependents_No',
 'Dependents_Yes',
 'PhoneService_No',
 'PhoneService_Yes',
 'MultipleLines_No',
 'MultipleLines_No phone service',
 'MultipleLines_Yes',
 'InternetService_DSL',
 'InternetService_Fiber optic',
 'InternetService_No',
 'OnlineSecurity_No',
 'OnlineSecurity_No internet service',
 'OnlineSecurity_Yes',
 'OnlineBackup_No',
 'OnlineBackup_No internet service',
 'OnlineBackup_Yes',
 'DeviceProtection_No',
 'DeviceProtection_No internet service',
 'DeviceProtection_Yes',
 'TechSupport_No',
 'TechSupport_No internet service',
 'TechSupport_Yes',
 'StreamingTV_No',
 'StreamingTV_No internet service',
 'StreamingTV_Yes',
 'StreamingMovies_No',
 'StreamingMovies_No internet service',
 'StreamingMovies_Yes',
 'Contract_Month-to-month',
 'Contract_One year',
 'Contract_Two year',
 'PaperlessBilling_No',
 'PaperlessBilling_Yes',
 'PaymentMethod_Bank tran

In [49]:
selected_mask=selector.get_support()
selected_mask

array([ True,  True,  True,  True,  True,  True,  True,  True, False,
        True, False, False,  True, False,  True, False,  True, False,
        True, False, False,  True, False,  True,  True, False, False,
        True, False, False, False, False, False, False, False, False,
        True, False,  True,  True,  True, False,  True,  True, False])

In [51]:
selected_features=np.array(all_features_names)[selected_mask]



In [46]:
print("\n Selected Features Names:")
for f in selected_features:
    print(f)


 Selected Features Names:
SeniorCitizen
tenure
MonthlyCharges
TotalCharges
gender_Female
gender_Male
Partner_No
Partner_Yes
Dependents_Yes
MultipleLines_No
MultipleLines_Yes
InternetService_Fiber optic
OnlineSecurity_No
OnlineBackup_No
OnlineBackup_Yes
DeviceProtection_No
TechSupport_No
Contract_Month-to-month
Contract_Two year
PaperlessBilling_No
PaperlessBilling_Yes
PaymentMethod_Credit card (automatic)
PaymentMethod_Electronic check
